# ML-07 — Baseline Action Score and Top-10 Review

The baseline is a transparent rule for prioritizing content review. It is frozen before the learned model: **stale (≥180 days) + visible (≥3,000 impressions in 90 days)**. The score is impressions for flagged pages and zero otherwise.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
paths=[Path('data/raw/content_refresh_anonymized.csv'),Path('../../data/raw/content_refresh_anonymized.csv')]
p=next((x for x in paths if x.exists()),None)
if p is None: raise FileNotFoundError('Starter dataset not found.')
df=pd.read_csv(p)
required={'content_id','days_since_last_update','impressions_90d','avg_position','trend_direction'}
missing=required-set(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'
# Evaluation-only proxy: never used by the rule itself.
df['is_declining_label']=(df['trend_direction'].astype(str).str.lower()=='down').astype(int)
print('Shape:',df.shape,'| Declining proxy rate:',round(df['is_declining_label'].mean(),3))

## 1. Signal checks and rule reasoning

**Signal A:** older pages may deserve review. **Signal B:** pages with meaningful visibility may be higher-value opportunities. Both are checked retrospectively, but neither uses the label to construct the score.

In [ ]:
age_bins=[-np.inf,30,90,180,365,np.inf]; age_labels=['0-30','31-90','91-180','181-365','365+']
df['freshness_bucket']=pd.cut(df['days_since_last_update'],bins=age_bins,labels=age_labels,include_lowest=True)
fresh_check=df.groupby('freshness_bucket',observed=False)['is_declining_label'].agg(n='size',declining_rate='mean')
vol_bins=[-np.inf,0,100,1000,3000,30000,np.inf]; vol_labels=['0','1-100','101-1k','1k-3k','3k-30k','30k+']
df['volume_bucket']=pd.cut(df['impressions_90d'],bins=vol_bins,labels=vol_labels,include_lowest=True)
volume_check=df.groupby('volume_bucket',observed=False)['is_declining_label'].agg(n='size',declining_rate='mean')
print('Freshness check'); print(fresh_check.round(3).to_string())
print('\nVisibility check'); print(volume_check.round(3).to_string())

## 2. Build the ranked queue

The rule has no fitted weights and carries a reason code and action label so a human can understand why an item was selected.

In [ ]:
stale=df['days_since_last_update'].fillna(0).ge(180)
visible=df['impressions_90d'].fillna(0).ge(3000)
selected=stale&visible
df['score']=np.where(selected,df['impressions_90d'].fillna(0),0.0)
df['reason_code']=np.where(selected,'stale_but_visible','not_flagged')
df['action_label']=np.where(selected,'refresh_review','monitor')
queue=df[['content_id','score','reason_code','action_label']].sort_values(['score','content_id'],ascending=[False,True]).reset_index(drop=True)
out=Path('work/outputs/baseline_action_score.csv')
out.parent.mkdir(parents=True,exist_ok=True)
queue.to_csv(out,index=False)
print('Rows ranked:',len(queue)); print('Flagged:',int(selected.sum())); print('Output:',out.resolve()); print(queue.head(10).to_string(index=False))

## 3. Retrospective evaluation

The label is reconstructed from `trend_direction` only for evaluation of this frozen rule. It is not an input to the score.

In [ ]:
def precision_at_k(scores,labels,k):
    order=np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())
base_rate=df['is_declining_label'].mean()
print('Base rate:',round(base_rate,3))
for k in [20,50]: print(f'Precision@{k}:',round(precision_at_k(df['score'],df['is_declining_label'],k),3))

## 4. Top-10 review

Each recommendation has an action, reason, and a concrete condition that could make the recommendation wrong.

In [ ]:
top10=queue.head(10).merge(df[['content_id','days_since_last_update','impressions_90d','avg_position']],on='content_id',how='left')
for i,row in top10.iterrows():
    wrong='Wrong if the page is already accurate/current, visibility is not commercially useful, or the measurement is unreliable.'
    print(f"{i+1}. {row['content_id']} | action={row['action_label']} | reason={row['reason_code']} | age={row['days_since_last_update']}d | impressions={row['impressions_90d']} | wrong_if={wrong}")

## 5. Leakage self-check

The rule inputs are only `days_since_last_update` and `impressions_90d`. Target-derived fields are explicitly forbidden.

In [ ]:
rule_inputs={'days_since_last_update','impressions_90d'}
forbidden={'trend_direction','trend_pct','is_declining_label'}
assert rule_inputs.isdisjoint(forbidden)
assert out.exists()
print('LEAKAGE CHECK: PASS')
print('Rule inputs:',sorted(rule_inputs))
print('Queue file exists:',out.exists())

## Self-check

- [x] Two signal checks with visible bucket counts.
- [x] Transparent frozen rule with score, reason code, and action label.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Base rate and Precision@20/50 are computed.
- [x] Top-10 recommendations include action, reason, and wrong-pick condition.
- [x] Label is evaluation-only and never enters score construction.
- [x] No client names, URLs, or private queries are displayed.